<img src="https://raw.githubusercontent.com/mjcowley/astrolabs/main/Images/qarg_bw_dark.png" alt="QUT Astrophysics" width="350"/>

# AstroLab C10: Gravity's Fingerprints

**PVB220 Cosmology** | Dr Michael Cowley | michael.cowley@qut.edu.au

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mjcowley/astrolabs/blob/main/AstroLabC10.ipynb)

## Introduction

Gravity is invisible, but it leaves fingerprints everywhere. In the last two lectures we saw three of them:

* **Bent light:** mass curves spacetime, so light passing a galaxy or cluster is deflected (gravitational lensing).
* **Missing mass:** galaxies spin far too fast for the matter we can see (dark matter).
* **Ripples:** accelerating masses shake spacetime itself (gravitational waves).

In the first half of this lab you'll bend starlight around the Sun, size up Einstein rings and weigh the dark matter in a real galaxy. In the second half you'll dig the first gravitational wave ever detected out of real LIGO data, use two detectors to locate it, and weigh the black holes that made it.

Have Canvas open alongside this notebook and answer each 🎯 **Canvas checkpoint** as you go.

🔊 Part 5 and the stretch activity can play sounds. Headphones are a bonus, but everything also works on mute.

▶️ Run the cell below first.

In [ ]:
#@title ▶️ Run me first: setup
import os
import urllib.request

# Download the latest helper file (keep the existing copy if we're offline)
try:
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/mjcowley/astrolabs/main/astrolabs.py",
        "astrolabs.py")
except Exception:
    if not os.path.exists("astrolabs.py"):
        raise

import numpy as np                 # numbers and arrays
import pandas as pd                # tables of data
import matplotlib.pyplot as plt    # plotting
import astropy.units as u          # physical units
from astropy.constants import G, c, M_sun, R_sun   # physical constants
from IPython.display import Audio  # playing sounds
import importlib
import astrolabs as al             # helper functions for these labs
importlib.reload(al)               # use the version just downloaded

al.setup()

---
# Half 1: Bending Light and Missing Mass

# Part 1: Einstein vs Newton

Einstein's general relativity predicts that light passing a mass $M$ at a distance $b$ is bent by an angle

$$\alpha = \frac{4GM}{c^2 b}$$

Let's redo the lecture's worked example: a ray of starlight that just grazes the edge of the Sun.

In [ ]:
alpha = 4 * G * M_sun / (c**2 * R_sun)

print("Deflection in radians:   ", alpha.decompose())
print("Deflection in arcseconds:", (alpha.decompose() * u.rad).to(u.arcsec))
al.check("C10_deflection_sun", (alpha.decompose() * u.rad).to(u.arcsec))

💡 The angle comes out **dimensionless**: all the units cancel, just as they did on the lecture slide. Astronomers measure angles in radians, then convert to arcseconds (1″ is 1/3600 of a degree).

> 🎯 **Canvas checkpoint Q1:** By how much does the Sun bend starlight that grazes its edge?

## Testing Einstein in 1919

Newton's gravity also bends light (if you treat light as tiny particles), but by **exactly half** as much. In 1919, Arthur Eddington's team photographed stars near the Sun during a total eclipse to find out who was right.

▶️ Run the cell below. Move the slider to see how the bending drops off with distance from the Sun, and tick the box to compare Newton's prediction.

In [ ]:
al.deflection_explorer()

> 🎯 **Canvas checkpoint Q2:** What deflection does **Newton's** gravity predict for starlight grazing the Sun, and did Eddington's 1919 measurement agree better with Newton or Einstein?

# Part 2: Einstein Rings

When a background source sits directly behind a lens, its light is smeared into a ring of angular radius

$$\theta_E = \sqrt{\frac{4GM}{c^2}\,\frac{D_{LS}}{D_L D_S}}$$

where $D_L$, $D_S$ and $D_{LS}$ are the distances to the lens, to the source, and from the lens to the source. Here it is as a Python **function** we can reuse:

In [ ]:
def einstein_radius(mass, d_lens, d_source, d_lens_source):
    theta = np.sqrt(4 * G * mass / c**2 * d_lens_source / (d_lens * d_source))
    return (theta.decompose() * u.rad).to(u.arcsec)

# A big galaxy acting as a lens, halfway to a distant source
print("Galaxy lens: ", einstein_radius(1e12 * M_sun, 1 * u.Gpc, 2 * u.Gpc, 1 * u.Gpc))

> 🎯 **Canvas checkpoint Q3:** What's the Einstein radius of the galaxy lens (10¹² M☉, halfway to a source 2 Gpc away)?

👉 Now use the function for a **galaxy cluster**, which is about 1,000 times more massive ($10^{15}\,M_\odot$). Change the mass in the cell below and run it.

In [ ]:
# 👉 Change the mass to 1e15 for a galaxy cluster
cluster_mass = 1e12 * M_sun

print("Cluster lens:", einstein_radius(cluster_mass, 1 * u.Gpc, 2 * u.Gpc, 1 * u.Gpc))

> 🎯 **Canvas checkpoint Q4:** When the lens is 1,000 times more massive, how much bigger is the Einstein ring?

💡 The ring grows as $\sqrt{M}$, so 1,000 times the mass gives about $\sqrt{1000} \approx 32$ times the radius. That's why clusters like SMACS 0723 stretch galaxies into huge arcs, while galaxy lenses like B1938+666 make rings a couple of arcseconds across.

🔍 To see arcs, multiple images and Einstein rings form, try the [Gravitational Lensing demo](https://mjcowley.github.io/astro_demos/lensing/): drag the background galaxy behind the lens.

# Part 3: Weighing the Dark Matter in NGC 2403

**NGC 2403** is a spiral galaxy about 10 million light-years away (the same galaxy on the lecture's dark matter slides). Radio astronomers measured how fast its hydrogen gas orbits at different distances from the centre: its **rotation curve**.

The table also lists how fast the gas would orbit if the galaxy's **visible** matter were all there is: the pull of its **stars** (estimated from their infrared light) and its **gas** (from radio maps).

In [ ]:
galaxy = pd.read_csv(al.get_data("C10/ngc2403_rotation_curve.csv"))

galaxy.head()

In [ ]:
# The speed from all visible matter combines the stars and gas
galaxy["v_visible"] = np.sqrt(galaxy["v_stars"]**2 + galaxy["v_gas"]**2)

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.errorbar(galaxy["radius_kpc"], galaxy["v_obs"], yerr=galaxy["v_err"], fmt="o", ms=4, color="k",
            label="Measured")
ax.plot(galaxy["radius_kpc"], galaxy["v_visible"], color=al.COLOURS["blue"], lw=2,
        label="Predicted from visible matter")
ax.set_xlabel("Distance from the centre (kpc)")
ax.set_ylabel("Rotation speed (km/s)")
ax.set_ylim(0, 160)
ax.legend()
plt.show()

outer = galaxy.iloc[-1]
print(f"At {outer['radius_kpc']:.0f} kpc: measured {outer['v_obs']:.0f} km/s, "
      f"visible matter predicts only {outer['v_visible']:.0f} km/s")

Just like the lecture's comparison of a Keplerian and a galactic rotation curve: the visible matter's prediction **falls away** in the outskirts, but the measured speeds stay **flat**. Something we can't see is adding extra gravity.

> 🎯 **Canvas checkpoint Q5:** At the outer edge of the data (about 21 kpc), how does the measured rotation speed compare with the prediction from visible matter?

## Add a dark matter halo

Now add a **dark matter halo**: a huge, roughly spherical cloud of invisible matter around the galaxy. It has two settings: its **speed** (how strongly it pulls far from the centre) and its **core** (how spread out its centre is).

▶️ Run the cell below and adjust the sliders until the red curve runs through the data, making the mismatch as small as you can.

💡 Even the best smooth model can't follow every bump in a real galaxy, so the smallest mismatch possible here is a little under 4.

In [ ]:
al.rotation_curve_fitter(galaxy)

> 🎯 **Canvas checkpoint Q6:** What halo speed gives the best fit?
>
> 🎯 **Canvas checkpoint Q7:** With your best fit, roughly what share of NGC 2403's mass (within 21 kpc) is dark matter?

> 🔍 **Peek inside:** the halo's shape comes from `halo_speed` in `astrolabs.py`. The speeds from different ingredients add **in quadrature**: $v_{\rm total}^2 = v_{\rm stars}^2 + v_{\rm gas}^2 + v_{\rm halo}^2$, because it's the masses (which go as $v^2$) that add up.

In [ ]:
# 👉 Optional self-check: enter your best halo speed in km/s
my_halo_speed = None
al.check("C10_halo_speed", my_halo_speed)

---
# Half 2: Ripples in Spacetime

# Part 4: How Small Is a Gravitational Wave?

A gravitational wave stretches and squeezes space by a tiny fraction called the **strain**, $h \approx \Delta L / L$. The lecture estimated it for two black holes (20 M☉ in total) orbiting so close they're nearly touching, 340 Mpc away:

$$h \approx \frac{GM}{c^2}\,\frac{1}{D}\left(\frac{v}{c}\right)^2$$

In [ ]:
M = 20 * M_sun          # combined mass
D = 1e25 * u.m          # 340 Mpc
v = 2e8 * u.m / u.s     # orbital speed when nearly touching (from the lecture)

h = (G * M / c**2) * (1 / D) * (v / c)**2
print(f"Strain h = {h.decompose():.1e}")
al.check("C10_strain", h.decompose())

💡 Notice the units cancel completely: strain is a pure number (a fraction), not a length.

So how much does a LIGO detector actually move? Each LIGO arm is 4 km long, and the change in length is $\Delta L = h \times L$.

In [ ]:
arm = 4 * u.km
change = h.decompose() * arm
proton = 1.7e-15 * u.m

print(f"Change in a 4 km LIGO arm: {change.to(u.m):.1e}")
print(f"That's about 1/{(proton / change).decompose():.0f} of the width of a proton!")

> 🎯 **Canvas checkpoint Q8:** By how much does a gravitational wave like this change the length of a 4 km LIGO arm?

# Part 5: Finding GW150914

On 14 September 2015, both LIGO detectors recorded the first gravitational wave ever detected, **GW150914**. Let's load the real data: 32 seconds from each detector, measured 4,096 times per second.

In [ ]:
ligo = np.load(al.get_data("C10/gw150914_strain.npz"))
fs = int(ligo["fs"])                                  # samples per second
hanford = ligo["H1"].astype(float)
livingston = ligo["L1"].astype(float)
time = np.arange(len(hanford)) / fs + float(ligo["t_start"])   # seconds from the event

print("Samples per second:", fs)
print("Number of samples: ", len(hanford))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(time, hanford, color=al.COLOURS["blue"], lw=0.5)
ax.set_xlabel("Time from the event (seconds)")
ax.set_ylabel("Strain")
ax.set_title("Raw LIGO Hanford data: where's the gravitational wave?")
plt.show()

The raw data is dominated by **noise**: ground vibrations, waves on distant beaches, passing trucks and the detector's own electronics. The real signal is hiding inside, about 100 times smaller.

Astronomers clean the data in two steps:

1. **Whitening:** the noise is much louder at some frequencies than others. Whitening divides out the noise at each frequency, so no frequency drowns out the rest.
2. **Band-pass filtering:** keep only frequencies between 35 and 350 Hz, where black-hole mergers like this one sing.

In [ ]:
hanford_white = al.whiten(hanford, fs)
hanford_clean = al.bandpass(hanford_white, fs, 35, 350)

livingston_white = al.whiten(livingston, fs)
livingston_clean = al.bandpass(livingston_white, fs, 35, 350)

▶️ Now step through the cleaning. Start at **1. Raw data**, then try each step. Use the slider to zoom in or out in time.

In [ ]:
al.gw_cleaner(time, hanford, hanford_white, hanford_clean)

> 🔍 **Peek inside:** `whiten` and `bandpass` in `astrolabs.py` use the `scipy.signal` package, the same tools LIGO scientists use.

## Seeing the chirp

A **spectrogram** shows how loud each frequency is at each moment, like sheet music for the signal.

In [ ]:
from scipy import signal

segment = (time > -1) & (time < 0.5)
freqs, times, power = signal.spectrogram(hanford_clean[segment], fs, nperseg=128, noverlap=124, nfft=1024)

fig, ax = plt.subplots(figsize=(10, 5))
ax.pcolormesh(times + time[segment][0], freqs, power, shading="auto", cmap="viridis")
ax.set_xlim(-0.3, 0.05)
ax.set_ylim(20, 400)
ax.set_xlabel("Time from the event (seconds)")
ax.set_ylabel("Frequency (Hz)")
ax.set_title("GW150914 in LIGO Hanford")
plt.show()

> 🎯 **Canvas checkpoint Q9:** What happens to the frequency of the gravitational wave in the final fraction of a second?

As the black holes spiral together they orbit faster and faster, so the wave's frequency **rises**: a "chirp".

## 🔊 Listen (optional)

The frequencies of GW150914 happen to be in the range of human hearing. Turn your sound on and play the cleaned data. Listen for a short "whoop" near the end.

In [ ]:
clip = (time > -1.5) & (time < 0.5)
Audio(hanford_clean[clip], rate=fs)

# Part 6: Two Detectors, One Source

A single detector can't tell where a wave came from, much like one ear struggles to locate a sound. LIGO has two detectors: **Hanford** (Washington) and **Livingston** (Louisiana), about 3,000 km apart. Gravitational waves travel at the speed of light, so the wave reaches one detector slightly before the other.

There's one twist: the two detectors' arms point in different directions, so Livingston sees the signal **upside down** compared with Hanford.

▶️ Run the cell below. Tick **flip**, then slide Livingston's signal in time until it lines up with Hanford's and the match is as high as possible.

In [ ]:
al.time_delay_explorer(time, hanford_clean, livingston_clean)

> 🎯 **Canvas checkpoint Q10:** How much later (in milliseconds) did the wave reach Hanford than Livingston?

The **largest** possible delay is the time light takes to cross the 3,000 km between the detectors. Let's check:

In [ ]:
separation = 3000 * u.km
max_delay = (separation / c).to(u.ms)
print(f"Largest possible delay: {max_delay:.1f}")

In [ ]:
# 👉 Optional self-check: your delay in milliseconds
my_delay = None
al.check("C10_delay_ms", my_delay)

> 🎯 **Canvas checkpoint Q11:** The measured delay (about 7 ms) is less than the 10 ms maximum. What does that tell us?

With a third and fourth detector (Virgo in Italy and KAGRA in Japan), astronomers can now triangulate sources much more precisely.

# Part 7: Weighing Black Holes

How quickly the frequency rises depends on the masses: heavier objects chirp faster and end at lower frequencies. The combination that matters most is the **chirp mass**. To leading order, the frequency a time $\tau$ before merger is

$$f(\tau) = \frac{1}{\pi}\left(\frac{5}{256\,\tau}\right)^{3/8}\left(\frac{G\mathcal{M}}{c^3}\right)^{-5/8}$$

▶️ Run the cell below and adjust the chirp mass until the dashed white curve follows the bright chirp in the spectrogram.

In [ ]:
al.chirp_mass_fitter(time, hanford_clean)

> 🎯 **Canvas checkpoint Q12:** Roughly what chirp mass matches GW150914?

💡 The chirp mass you measure (about 30 M☉) is a little larger than the catalogue value (28.6 M☉), because the expanding Universe also stretched the signal: the same redshift you met in AstroLab C8!

## Where did the mass go?

The LIGO catalogue gives the full answer. Let's load every confirmed merger so far.

In [ ]:
catalogue = pd.read_csv(al.get_data("C10/gw_catalogue.csv"))

print("Confirmed mergers:", len(catalogue))
catalogue[catalogue["event"] == "GW150914"]

In [ ]:
gw = catalogue[catalogue["event"] == "GW150914"].iloc[0]

missing = (gw["mass_1"] + gw["mass_2"] - gw["final_mass"]) * M_sun
energy = (missing * c**2).to(u.J)

print(f"Before: {gw['mass_1']} + {gw['mass_2']} = {gw['mass_1'] + gw['mass_2']:.1f} M☉")
print(f"After:  {gw['final_mass']} M☉")
print(f"Mass turned into gravitational waves: {missing.to(u.M_sun):.1f}")
print(f"Energy, E = mc²: {energy:.1e}")

sun_lifetime_output = (3.8e26 * u.W * 10e9 * u.yr).to(u.J)
print(f"That's {(energy / sun_lifetime_output).decompose():,.0f} times all the energy the Sun will give out in its 10-billion-year life!")

> 🎯 **Canvas checkpoint Q13:** How much mass did GW150914 turn into gravitational waves?

## The whole population

Since 2015, LIGO, Virgo and KAGRA have detected hundreds of mergers.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(catalogue["mass_1"], catalogue["mass_2"], s=20, alpha=0.6, color=al.COLOURS["blue"])
for name, colour in [("GW150914", al.COLOURS["red"]), ("GW170817", al.COLOURS["orange"]),
                     ("GW190521", al.COLOURS["teal"])]:
    one = catalogue[catalogue["event"] == name]
    ax.scatter(one["mass_1"], one["mass_2"], s=120, color=colour, edgecolor="k", label=name, zorder=3)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Heavier object (M☉)")
ax.set_ylabel("Lighter object (M☉)")
ax.legend()
plt.show()

> 🎯 **Canvas checkpoint Q14:** Which highlighted event was the merger of two **neutron stars**, and how can you tell from the plot?

**GW170817** was two neutron stars, each about 1.4 M☉. It was also seen as a gamma-ray burst and a "kilonova" by telescopes around the world, the first event seen in both gravitational waves and light. **GW190521**, at the other extreme, formed a black hole of about 150 M☉.

---
# What's Coming Up

| Lab | Week | What you'll do |
| --- | :-: | --- |
| C7 | 2 | Python toolkit |
| C8 | 4 | The Big Bang and the CMB |
| C9 | 6 | Galaxies and their black holes |
| **C10** | 8 | Lensing, dark matter and gravitational waves (this lab!) |
| **C11** | 10 | Cosmology surveys: cross-matching catalogues and scaling relations |

### Sparkler ideas from this lab

* The 1919 eclipse: how Einstein beat Newton
* Weighing the invisible: dark matter in NGC 2403
* Finding a needle in a haystack: digging GW150914 out of the noise
* Hundreds of mergers: what the gravitational wave population tells us

**Well done, you've finished AstroLab C10!** Make sure you've answered all the 🎯 Canvas checkpoints.

---
# 🚀 Stretch Activity: Build Your Own Chirp

*Optional, if you have time left.*

Here's a simple chirp built from just the chirp mass. 🔊 Play the black-hole chirp, then change the chirp mass to **1.2 M☉**, about right for two neutron stars like GW170817, and play it again.

In [ ]:
# 👉 Try 28 (GW150914-like black holes), then 1.2 (neutron stars)
my_chirp_mass = 28

t, wave = al.chirp_sound(my_chirp_mass, duration=20)

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(t, wave, color=al.COLOURS["blue"], lw=0.5)
ax.set_xlabel("Time before merger (seconds)")
ax.set_title(f"Chirp for a chirp mass of {my_chirp_mass} M☉ (starting at 30 Hz)")
plt.show()

print(f"Time spent above 30 Hz: {-t[0]:.2f} seconds")
Audio(wave, rate=4096)

Lighter objects spend much longer in LIGO's frequency range: a black-hole pair whoops for a fraction of a second, while a neutron-star pair chirps for many seconds (GW170817 was in band for about 100 seconds from 24 Hz).

🚀 **Extra challenge:** the chirp ends when the two objects touch. Why would two neutron stars reach a **higher** final frequency than two black holes? (Hint: think about how big each object is compared with its mass.)

---
### Data and image credits

* NGC 2403 rotation curve and mass models: SPARC, Lelli, McGaugh & Schombert 2016, AJ, 152, 157, via VizieR. Star speeds assume a typical 3.6 μm mass-to-light ratio of 0.5
* Dark matter halo model: pseudo-isothermal sphere
* GW150914 strain data and the event catalogue: Gravitational Wave Open Science Center (gwosc.org); LIGO Scientific Collaboration, Virgo Collaboration and KAGRA Collaboration. GW150914: Abbott et al. 2016, PRL, 116, 061102
* Eddington's 1919 measurement: Dyson, Eddington & Davidson 1920, Phil. Trans. R. Soc. A, 220, 291 (Sobral plates)